# 01 — PyTorch Fundamentals

> **Stage 1 (Baselines)** — first executable notebook of the repo.

We load a HuggingFace model, generate sentence embeddings, and compute cosine similarity manually with PyTorch. This is the smoke test of the stack: if everything here runs cleanly, every embedder we benchmark in this repo (OpenAI, Voyage, BGE-M3, Jina/Qwen3, fine-tuned BGE-M3) is just a swap of the model id.

**What we cover**

1. Tensor basics + device selection (MPS / CUDA / CPU)
2. Loading a tokenizer + model with `transformers`
3. Inference with `torch.no_grad()` + `model.eval()`
4. Mean pooling with attention mask
5. L2 normalization + manual cosine similarity
6. The MPS warmup gotcha (first inference is 5–10× slower)

**Smoke-test model**: `BAAI/bge-small-en-v1.5` (33M params, ~130 MB download). Lightweight on purpose — the real Stage 1 baselines (OpenAI `text-embedding-3-large`, BGE-M3) come in later notebooks. The recipe (tokenize → forward → mean pool → L2 norm) transfers directly.

## 1. Imports

Three libraries: `torch` for tensors, `torch.nn.functional` for `normalize`, and `transformers` for the auto-classes. `time` is for the warmup demo at the end.

In [ ]:
import time

import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

print(f"PyTorch       : {torch.__version__}")
print(f"MPS available : {torch.backends.mps.is_available()}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Device detection

We pick **MPS** if running on Apple Silicon, **CUDA** if running on a NVIDIA box, otherwise **CPU**. The order matters: in a Mac with both available (rare), we still want MPS — it's the GPU on the user's machine.

> **Heads up**: MPS doesn't support `float64`. PyTorch defaults to `float32` on MPS, so this is rarely a problem in practice — but if you load NumPy arrays with `dtype=float64` and move them to MPS, cast first.

In [ ]:
def get_device() -> str:
    """Return the best available device: MPS > CUDA > CPU."""
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


device = get_device()
print(f"Using device: {device}")

## 3. Tensor basics

A `torch.Tensor` is like a NumPy `ndarray` with three superpowers: it can live on a GPU, it tracks gradients (autograd, opt-in), and it integrates with PyTorch's neural network layers. Same indexing, same broadcasting, same vectorized ops — if you know NumPy, you know 90% of tensors.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
print(f"x        = {x}")
print(f"x.shape  = {x.shape}")
print(f"x.dtype  = {x.dtype}")
print(f"x.device = {x.device}")

x_dev = x.to(device)
print(f"\nAfter .to({device!r}):")
print(f"x_dev.device = {x_dev.device}")

## 4. Load tokenizer + model

We use `BAAI/bge-small-en-v1.5` as a smoke test:

- 33M params, ~130 MB download — the first run pulls weights from the Hub; subsequent runs read from `~/.cache/huggingface/hub/`.
- Same tokenizer family and pooling recipe as BGE-M3 (used later in this Stage), so what you learn here transfers.
- This is **not** the embedder we'll benchmark — it's a toy choice to validate the stack. The real Stage 1 baselines come in `02_openai_baseline.ipynb` and `03_bge_m3_baseline.ipynb`.

> **Gotcha — `HF_TOKEN` warning**: `transformers` will print a warning about `HF_TOKEN` not being set. It's defensive — public models like this one don't need a token. Ignore it. The warning matters only for gated or private repos.

In [ ]:
MODEL_ID = "BAAI/bge-small-en-v1.5"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device).eval()

print(f"Model: {MODEL_ID}")
print(f"Hidden size            : {model.config.hidden_size}")
print(f"Max position embeddings: {model.config.max_position_embeddings}")
print(f"Vocab size             : {tokenizer.vocab_size}")

## 5. Tokenize a batch

Three sentences: two are semantically related (Apple earnings), the third is unrelated. The tokenizer:

- splits each sentence into subword tokens
- pads shorter sequences to match the longest in the batch (`padding=True`)
- truncates anything past the model's max length (`truncation=True`)
- returns PyTorch tensors directly (`return_tensors="pt"`)

The `attention_mask` is the bookkeeping that tells the model (and us, later) which tokens are real and which are padding.

In [ ]:
sentences = [
    "Apple reported record revenue in fiscal Q4.",
    "iPhone sales drove Apple's quarterly earnings.",
    "The cat sat on the mat.",
]

batch = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    return_tensors="pt",
).to(device)

print(f"input_ids.shape      = {tuple(batch['input_ids'].shape)}      # (batch, seq_len)")
print(f"attention_mask.shape = {tuple(batch['attention_mask'].shape)} # (batch, seq_len)")
print(f"\nFirst sentence input_ids[:12]      = {batch['input_ids'][0, :12].tolist()}")
print(f"First sentence attention_mask[:12] = {batch['attention_mask'][0, :12].tolist()}")

## 6. Inference with `torch.no_grad()`

Two flags matter for inference, and they are **not the same**:

| Flag | What it does | When |
|---|---|---|
| `model.eval()` | Disables dropout and switches batch-norm to running stats | Set once after loading |
| `torch.no_grad()` | Disables autograd graph construction | Wrap each inference call |

We already called `model.eval()` when loading. Now we wrap the forward pass in `torch.no_grad()` — less memory, faster, no risk of accidentally building a gradient graph for tensors we don't plan to backprop through.

In [ ]:
with torch.no_grad():
    outputs = model(**batch)

last_hidden = outputs.last_hidden_state
print(f"last_hidden.shape = {tuple(last_hidden.shape)}  # (batch, seq_len, hidden_dim)")

## 7. Mean pooling

`last_hidden_state` has one vector per token. To get one vector per sentence, we average across the `seq_len` dimension — but we use `attention_mask` to ignore padding tokens. If we just averaged blindly, padding positions (which carry near-zero contextual signal) would dilute the real content.

> **Why mean pooling and not CLS pooling?** Different models train with different recipes. BGE-small uses mean pooling. BGE-M3 (next notebook) uses CLS for the dense vector — always check the model card. The mean-pooling helper below is reused as-is for any model that documents mean pooling.

In [ ]:
def mean_pooling(
    token_embeddings: torch.Tensor,
    attention_mask: torch.Tensor,
) -> torch.Tensor:
    """Average token embeddings along seq_len, weighted by attention_mask."""
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    summed = (token_embeddings * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


embeddings = mean_pooling(last_hidden, batch["attention_mask"])
print(f"embeddings.shape = {tuple(embeddings.shape)}  # (batch, hidden_dim)")

## 8. L2 normalization + cosine similarity

Cosine similarity is `(a · b) / (‖a‖ · ‖b‖)`. If we normalize `a` and `b` to unit length, the denominators collapse to 1 and the formula reduces to a plain dot product — faster in batch, mathematically identical, and the result lives in `[-1, 1]`.

This is the same identity that powers retrieval at scale: vector databases store **normalized** vectors so that nearest-neighbor search by dot product is equivalent to nearest-neighbor by cosine similarity.

After normalization we expect each row of `embeddings` to have norm 1. We verify, then compute the full pairwise similarity matrix with a single matmul.

In [ ]:
embeddings = F.normalize(embeddings, p=2, dim=1)
print(f"Norms after normalize: {[round(n, 4) for n in embeddings.norm(dim=1).tolist()]}")

sim_matrix = embeddings @ embeddings.T
print("\nCosine similarity matrix:")
print(sim_matrix.cpu().numpy().round(3))

**Reading the matrix**: the diagonal is always 1.0 (each sentence vs itself). Off-diagonal entries `[0,1]` and `[1,0]` should be high — both sentences are about Apple's earnings — while everything involving sentence 2 ("the cat...") should be markedly lower. If the matrix matches that pattern, the embedder is doing what it's supposed to.

## 9. The MPS warmup gotcha

The very first inference call on MPS triggers JIT compilation of Metal kernels — it can be 5–10× slower than subsequent runs. The same effect exists on CUDA but is much less pronounced.

**Always discard the first measurement** when timing inference on MPS. It's the kernel cache warming up, not your machine being slow. We synchronize the device before reading the clock so we're measuring real GPU work, not just kernel dispatch.

In [ ]:
def time_inference() -> float:
    """Return milliseconds for a single forward pass over the current batch."""
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(**batch)
    if device == "mps":
        torch.mps.synchronize()
    elif device == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - start) * 1000


for i in range(1, 6):
    label = "warmup, ignore" if i == 1 else ""
    print(f"Run {i}: {time_inference():6.1f} ms  {label}")

## What's next

You now have every primitive needed to build the embedder wrappers used across this repo:

| Concept | Reused in |
|---|---|
| `AutoTokenizer` + `AutoModel` | All HuggingFace embedders (`src/embeddings/`) |
| Mean pooling + L2 norm | BGE-small here, BGE-M3 in `03_bge_m3_baseline` |
| `torch.no_grad()` + `model.eval()` | Every embedder wrapper |
| Cosine similarity matrix | Retrieval scoring across all baselines |

Next notebook: load the FinanceBench dataset and turn 10-K filings into chunks ready to embed (`paso-03` in the course, `02_financebench_loader` here).